# FarmTech Solutions — Fase 6: Visão Computacional
## Entrega 1 — Detecção de Objetos com YOLO Customizado

**Grupo:** Grupo 3
**Integrantes:** Gerson Ferreira da Graça (RM 569624), Lucas Braga (RM 568712), Carlos Leonardo Mazieri (RM 572809), Ryann Pinto (RM 571003)
**Objetos escolhidos:** Tomate (Objeto A) e Pimentão (Objeto B)

---

### Objetivo desta entrega

Treinar um modelo YOLO customizado para detectar e diferenciar tomates e pimentões em imagens, comparando o desempenho com duas configurações diferentes de épocas de treino (30 e 60).


## 1. Setup do ambiente

Conectando ao Google Drive e instalando as dependências necessárias.

In [ ]:
# Conectar ao Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Caminho base do projeto no Drive
BASE_PATH = '/content/drive/MyDrive/FarmTech_Fase6'


In [ ]:
# Instalar YOLO (Ultralytics)
!pip install ultralytics -q

from ultralytics import YOLO
import os
import random
import shutil


## 2. Organização do Dataset

Nesta etapa, confirmamos que o dataset está corretamente dividido em treino (32 imagens), 
validação (4 imagens) e teste (4 imagens) para cada classe (tomate e pimentão), 
totalizando 80 imagens.


In [ ]:
# Verificar estrutura do dataset
tomate_treino = os.path.join(BASE_PATH, 'dataset_dividido/tomate/treino')
tomate_val = os.path.join(BASE_PATH, 'dataset_dividido/tomate/validacao')
tomate_teste = os.path.join(BASE_PATH, 'dataset_dividido/tomate/teste')

pimentao_treino = os.path.join(BASE_PATH, 'dataset_dividido/pimentao/treino')
pimentao_val = os.path.join(BASE_PATH, 'dataset_dividido/pimentao/validacao')
pimentao_teste = os.path.join(BASE_PATH, 'dataset_dividido/pimentao/teste')

print(f"Tomate - Treino: {len(os.listdir(tomate_treino))} imagens")
print(f"Tomate - Validação: {len(os.listdir(tomate_val))} imagens")
print(f"Tomate - Teste: {len(os.listdir(tomate_teste))} imagens")
print(f"Pimentão - Treino: {len(os.listdir(pimentao_treino))} imagens")
print(f"Pimentão - Validação: {len(os.listdir(pimentao_val))} imagens")
print(f"Pimentão - Teste: {len(os.listdir(pimentao_teste))} imagens")


### 2.1 Arquivo de configuração do dataset (data.yaml)

O YOLO precisa de um arquivo `.yaml` descrevendo onde estão as imagens e quais são as classes.


In [ ]:
# Criar o arquivo data.yaml necessário para o treino do YOLO
data_yaml_content = f"""
train: {BASE_PATH}/dataset_dividido/treino/images
val: {BASE_PATH}/dataset_dividido/validacao/images

nc: 2
names: ['tomate', 'pimentao']
"""

with open(f'{BASE_PATH}/data.yaml', 'w') as f:
    f.write(data_yaml_content)

print("Arquivo data.yaml criado.")
print(data_yaml_content)


> **Nota:** o YOLO espera as anotações (rótulos) no formato `.txt`, uma por imagem, 
> exportadas do Make Sense IA. Confirme que os arquivos de rótulo estão na estrutura 
> esperada (`images/` e `labels/` pareados) antes de prosseguir.

## 3. Rotulação das Imagens (Make Sense IA)

**Esta etapa é feita FORA do Colab**, diretamente no site [Make Sense IA](https://www.makesense.ai/).

Passo a passo:
1. Acesse makesense.ai
2. Carregue as imagens de treino
3. Crie as labels: `tomate` e `pimentao`
4. Desenhe as bounding boxes em cada imagem
5. Exporte no formato **YOLO** (gera um `.txt` por imagem)
6. Faça upload dos arquivos exportados para a pasta `rotulacoes/` no Drive


## 4. Treino do Modelo — Simulação 1 (30 épocas)

Primeira simulação, usando 30 épocas de treino.


In [ ]:
# Carregar modelo base YOLO (pré-treinado, para fazer fine-tuning)
model_30 = YOLO('yolov8n.pt')

# Treinar com 30 épocas
results_30 = model_30.train(
    data=f'{BASE_PATH}/data.yaml',
    epochs=30,
    imgsz=640,
    project=f'{BASE_PATH}/resultados/epocas_30',
    name='treino'
)


### 4.1 Avaliação da Simulação 1 (30 épocas)

In [ ]:
# Validar o modelo treinado com 30 épocas
metrics_30 = model_30.val()

print(f"mAP50: {metrics_30.box.map50}")
print(f"mAP50-95: {metrics_30.box.map}")


## 5. Treino do Modelo — Simulação 2 (60 épocas)

Segunda simulação, usando 60 épocas de treino, para comparação.


In [ ]:
# Carregar modelo base novamente (do zero, para comparação justa)
model_60 = YOLO('yolov8n.pt')

# Treinar com 60 épocas
results_60 = model_60.train(
    data=f'{BASE_PATH}/data.yaml',
    epochs=60,
    imgsz=640,
    project=f'{BASE_PATH}/resultados/epocas_60',
    name='treino'
)


### 5.1 Avaliação da Simulação 2 (60 épocas)

In [ ]:
# Validar o modelo treinado com 60 épocas
metrics_60 = model_60.val()

print(f"mAP50: {metrics_60.box.map50}")
print(f"mAP50-95: {metrics_60.box.map}")


## 6. Comparação entre as Simulações (30 vs 60 épocas)

| Métrica | 30 épocas | 60 épocas |
|---|---|---|
| mAP50 | *(preencher após rodar)* | *(preencher após rodar)* |
| mAP50-95 | *(preencher após rodar)* | *(preencher após rodar)* |
| Tempo de treino | *(preencher)* | *(preencher)* |

*(Preencher esta tabela com os valores reais obtidos acima, e discutir os resultados no texto abaixo.)*

### Análise crítica

*(Escrever aqui: qual configuração teve melhor desempenho? Houve overfitting em alguma? 
O ganho de mais épocas compensou o tempo adicional de treino? Justificar com base nos números 
obtidos acima.)*


## 7. Teste do Modelo (imagens nunca vistas)

Rodando o modelo escolhido (o de melhor desempenho entre as duas simulações) 
contra as imagens de teste, que não foram usadas nem no treino nem na validação.


In [ ]:
# Escolher o melhor modelo entre os dois treinados acima
# (ajustar a variável 'melhor_modelo' conforme o resultado da comparação)
melhor_modelo = model_60  # ou model_30, dependendo do resultado

# Rodar predição nas imagens de teste
resultados_teste = melhor_modelo.predict(
    source=f'{BASE_PATH}/dataset_dividido/teste/images',
    save=True,
    conf=0.5
)

print(f"Total de imagens testadas: {len(resultados_teste)}")


### 7.1 Prints dos resultados de teste

*(Inserir aqui os prints/imagens processadas pelo modelo, salvos automaticamente em 
`yolov5/runs/detect/expX` ou pasta equivalente do Ultralytics — confirmar caminho exato 
gerado pela célula acima antes de inserir.)*


## 8. Conclusões

### Pontos fortes do trabalho

*(Escrever: o que funcionou bem? O modelo diferenciou tomate e pimentão com boa precisão? 
A variedade de cores/ângulos ajudou a generalização?)*

### Limitações

*(Escrever: quais os limites encontrados? Dataset pequeno (80 imagens) pode ter afetado 
a robustez? Alguma classe teve mais dificuldade que outra?)*

### Aprendizados

*(Escrever: o que a equipe aprendeu sobre visão computacional, YOLO, e o processo de 
criação de dataset customizado?)*
